<div style="background-color: #FFFF00; color: #000000; padding: 12px 18px; font-family: Arial, Helvetica, sans-serif;">
<strong style="color: #000000;">MINE-4101 Ciencia de Datos Aplicada · Universidad de los Andes · 2026-20</strong><br>
<span style="color: #000000;">Proyecto final · Entrega final · Actividad 8</span>
</div>

# Preparación de los datos

Del archivo de predios a la tabla con la que se entrenan los modelos que estiman si un predio pagará el impuesto predial, para un municipio colombiano.

**Integrantes**

- Daniel Felipe Triviño Santana
- Nicolas Bedoya Figueroa
- Joubert Daniel Álvarez
- Gabriel Bejarano

**Qué pide el enunciado** (actividad 8, 20 % de la entrega final): describir el proceso de preparación y mostrar evidencia de los datos preparados, detallando las transformaciones (nuevas características, codificación de variables categóricas, normalización, estandarización, imputación de faltantes, etc.). Se recomienda un diagrama de bloques funcional de los procesos.

**Punto de partida:** el [notebook 06](../../entrega-1/notebooks/06-entendimiento-de-los-datos.ipynb) de la primera entrega: la unidad de análisis (etapa 3.4), los atributos derivados por predio (3.3) y los problemas de calidad con su tratamiento propuesto (tabla de la etapa 6.3). El modelo es el del enfoque analítico (sección 4 del [reporte](../reporte/reporte.pdf)): clasificación binaria de si el predio pagó la vigencia 2026, con variables conocidas antes de la vigencia.

**Dos tipos de transformación.** Las que no aprenden nada de los datos (limpieza, nuevas características, filtros) se hacen aquí y quedan en `datos/modelado/predios_preparados.parquet`. Las que sí aprenden de ellos (imputación con medianas, escalado, codificación con las categorías vistas) se ajustan solo con el conjunto de entrenamiento, dentro del pipeline del notebook 10, para no filtrar información de validación y prueba. Aquí se definen y se justifican.

**Cómo hacerlo:** el [sitio de prácticas del curso](https://datascience-uniandes.github.io/PracticasMachineLearning/) es la referencia principal; para esta actividad, [práctica 2: Clasificación](https://datascience-uniandes.github.io/PracticasMachineLearning/talleres/practica-2/) (actividad 2, ingeniería de características) y las guías [valores nulos](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/nulos/), [valores inválidos](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/valores-invalidos/), [características derivadas](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/caracteristicas-derivadas/), [one-hot](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/one-hot/), [estandarizar](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/estandarizar/), [escalar variables](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/escalar-variables/) y [pipeline](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/pipeline/). Si el notebook se aparta de ellas, lo dice y explica por qué.

**Estado:** las ocho etapas tienen código e interpretación. Pendiente de revisión del equipo.

## Tabla de contenido

| | Etapa | Qué se hace |
| --- | --- | --- |
| 1 | [Librerías y configuración](#librerias) | Librerías, rutas y estilo |
| 2 | [Carga de los datos](#carga) | Leer el archivo de predios |
| 3 | [Unidad de análisis y variable objetivo](#unidad) | Qué predios entran al modelo y qué se predice |
| 4 | [Limpieza](#limpieza) | Tratar los problemas de calidad de la etapa 6.3 del notebook 06 |
| 5 | [Nuevas características](#caracteristicas) | Variables derivadas, como el historial de pago con una categoría para los predios nuevos |
| 6 | [Codificación, escalado e imputación](#transformaciones) | Definir las transformaciones que se ajustan en entrenamiento |
| 7 | [Evidencia de los datos preparados](#evidencia) | Dimensiones, tipos, faltantes y una muestra; guardar el archivo |
| 8 | [Diagrama de bloques](#diagrama) | El flujo de la preparación |

<a id="librerias"></a>

## 1. Librerías y configuración

Las librerías están en el entorno del proyecto (ver el [README](../../README.md#entorno-de-python)). Las rutas de los archivos que pasan de un notebook a otro, la semilla y las columnas reservadas están en [utilidades_modelado.py](utilidades_modelado.py), junto a este notebook.

In [1]:
import platform

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
import sklearn

# Rutas, semilla, columnas reservadas y estilo (utilidades_modelado.py, en esta carpeta)
import utilidades_modelado as um

um.configurar()

In [2]:
# Versiones con las que se ejecutó el notebook
print(f"Python {platform.python_version()}")
for libreria in (np, pd, mpl, sns, sklearn):
    print(f"{libreria.__name__} {libreria.__version__}")

Python 3.12.15
numpy 2.3.5
pandas 2.3.3
matplotlib 3.10.0
seaborn 0.13.2
sklearn 1.9.1


<a id="carga"></a>

## 2. Carga de los datos

El archivo `datos/predios/predios.parquet` no está en el repositorio (ver [datos/predios/](../../datos/predios/README.md)). Se lee igual que en la etapa 2 del notebook 06: las estructuras (`terreno`, `titulares`) se abren en columnas con su prefijo y las listas (`construcciones`, `unidades`) se desanidan en dos tablas con el `id_predio` al que pertenecen.

**Salidas.** El repositorio es público: ninguna salida puede mostrar datos personales ni nada que identifique al municipio, y los valores de `um.COLUMNAS_RESERVADAS` no se muestran (ver la etapa 2 del [notebook 06](06-entendimiento-de-los-datos.ipynb)). La variable objetivo y las que salen de la facturación se muestran solo agregadas.

In [3]:
um.exigir(um.ARCHIVO_PREDIOS, "pídalo al equipo o constrúyalo como indica datos/predios/README.md")
predios = pq.read_table(um.ARCHIVO_PREDIOS).flatten().to_pandas()
predios["fecha_visita_predial"] = pd.to_datetime(predios["fecha_visita_predial"])


def desanidar(columna):
    # Una fila por elemento de una columna de listas, con el id del predio al que pertenece
    filas = predios[["id_predio", columna]].explode(columna).dropna(subset=[columna])
    return pd.concat([filas[["id_predio"]].reset_index(drop=True), pd.json_normalize(filas[columna].tolist())],
                     axis=1)


construcciones = desanidar("construcciones")
unidades = desanidar("unidades")
predios = predios.drop(columns=["construcciones", "unidades"])

pd.DataFrame({"filas": [len(predios), len(construcciones), len(unidades)],
              "columnas": [predios.shape[1], construcciones.shape[1], unidades.shape[1]]},
             index=["predios", "construcciones", "unidades"])

,filas,columnas
predios,22220,76
construcciones,13483,15
unidades,42988,21


<a id="unidad"></a>

## 3. Unidad de análisis y variable objetivo

**Unidad de análisis:** el predio facturado por Hacienda en la vigencia 2026, como en la etapa 3.4 del notebook 06. Dos filtros:

- **Con facturación 2026.** Sin facturación no hay variable objetivo. Los que faltan son matrices, unidades de parque cementerio y predios cancelados en la actualización (etapa 7 del notebook 06), así que el filtro no sesga la muestra.
- **Sin vías, bienes de uso público ni parque cementerio.** Suelen estar exentos y su pago no responde a lo mismo que el de un predio privado.

**Variable objetivo:** `pago_2026`, 1 si el predio pagó la vigencia 2026 a la fecha de extracción de la facturación y 0 si no. **Grupo:** `historial_2025` (pagó 2025, no pagó 2025, sin registro de 2025), que estratifica la partición (notebook 09) y separa las métricas (notebook 10).

Los filtros se aplican al final (etapa 7). Hasta entonces se trabaja con todo el levantamiento, porque las unidades de PH toman datos de su matriz, que no está facturada.

In [4]:
EXCLUIR = ["Via", "Bien_Uso_Publico", "Parque_Cementerio.Matriz", "Parque_Cementerio.Unidad_Predial"]
con_facturacion = predios["pago_2026"].notna()
en_unidad = con_facturacion & ~predios["condicion_predio"].isin(EXCLUIR)

display(pd.DataFrame({"predios": [len(predios), con_facturacion.sum(), en_unidad.sum()]},
                     index=["Levantamiento", "Con facturación 2026",
                            "Sin vías, bienes de uso público ni parque cementerio"]))

predios[um.OBJETIVO] = predios["pago_2026"].astype("boolean").astype("Int64")
predios[um.GRUPO] = predios["pago_2025"].map({True: "Pagó 2025", False: "No pagó 2025"}).fillna("Sin registro 2025")

# Solo cifras agregadas de la variable objetivo
objetivo = predios[en_unidad].groupby(um.GRUPO)[um.OBJETIVO].agg(predios="size", pct_pagaron="mean")
objetivo.loc["Total"] = [en_unidad.sum(), predios.loc[en_unidad, um.OBJETIVO].mean()]
objetivo.assign(pct_pagaron=lambda t: t["pct_pagaron"] * 100).round(1)

,predios
Levantamiento,22220
Con facturación 2026,19332
"Sin vías, bienes de uso público ni parque cementerio",19149


,predios,pct_pagaron
historial_2025,,
No pagó 2025,2703.00,0.30
Pagó 2025,12519.00,80.30
Sin registro 2025,3927.00,56.10
Total,19149.00,64.00


**Interpretación.** De los 22 220 predios del levantamiento, 19 332 tienen facturación de 2026 y 19 149 quedan en la unidad de análisis, después de quitar 182 vías y bienes de uso público y una matriz de parque cementerio. Pagó la vigencia 2026 el 64,0 %. Los grupos del historial son los del notebook 06:

- 12 519 pagaron 2025, y de ellos pagó 2026 el 80,3 %;
- 3 927 no tienen registro de 2025, y pagó el 56,1 %;
- 2 703 no pagaron 2025, y pagó el 0,3 %.

<a id="limpieza"></a>

## 4. Limpieza

El tratamiento de los problemas de la tabla de la etapa 6.3 del notebook 06 que afectan a las variables del modelo. Ninguno aprende de los datos de entrenamiento: son reglas fijas (un valor de relleno pasa a faltante, un valor incoherente se descarta), así que se aplican antes de partir los datos. La imputación de los faltantes que quedan va en el pipeline (etapa 6).

In [5]:
limpieza = []


def registrar(problema, afectados, tratamiento):
    limpieza.append({"problema": problema, "predios de la unidad": int(afectados), "tratamiento": tratamiento})


# Número de pisos igual a 99 (con altura de 297 m): valor de relleno
pisos_99 = construcciones["total_pisos"].eq(99)
construcciones["total_pisos"] = construcciones["total_pisos"].mask(pisos_99)
registrar("Número de pisos igual a 99", predios.loc[en_unidad, "id_predio"].isin(construcciones.loc[pisos_99, "id_predio"]).sum(),
          "Faltante")

# Fecha de visita repetida en bloque: la fecha más frecuente del levantamiento, todas con visita incompleta
fecha_centinela = predios["fecha_visita_predial"].value_counts().idxmax()
es_centinela = predios["fecha_visita_predial"].eq(fecha_centinela)
predios["fecha_visita_valida"] = predios["fecha_visita_predial"].mask(es_centinela)
registrar("Fecha de visita de relleno", (es_centinela & en_unidad).sum(),
          "Faltante; la visita cuenta como exitosa solo con fecha válida")

# Año de construcción: mediana de las unidades de cada predio; el año 2000 es un valor de relleno
anio = unidades.groupby("id_predio")["anio_construccion"].median()
predios["anio_construccion"] = predios["id_predio"].map(anio)
predios["anio_sospechoso"] = predios["anio_construccion"].eq(2000).astype(int)
registrar("Año de construcción 2000 de relleno", (predios["anio_sospechoso"].eq(1) & en_unidad).sum(),
          "Faltante, con el indicador anio_sospechoso")
predios["anio_construccion"] = predios["anio_construccion"].mask(predios["anio_sospechoso"].eq(1))

# Avalúo igual a 0: faltante, con indicador (Hacienda liquidó el impuesto con su propio avalúo)
predios["avaluo_cero"] = predios["avaluo_catastral"].le(0).astype(int)
registrar("Avalúo catastral igual a 0", (predios["avaluo_cero"].eq(1) & en_unidad).sum(),
          "Faltante, con el indicador avaluo_cero; tarifa efectiva faltante")

# Área registral: 0 o más de 20 veces menor que la construida (el título no describe lo levantado)
area_construida = unidades.groupby("id_predio")["area_construida"].sum()
predios["area_construida_total"] = predios["id_predio"].map(area_construida).fillna(0)
registral_invalida = predios["area_registral_m2"].le(0) | (predios["area_construida_total"] > 20 * predios["area_registral_m2"])
registrar("Área registral igual a 0 o más de 20 veces menor que la construida", (registral_invalida & en_unidad).sum(),
          "Faltante")
predios["area_registral_valida"] = predios["area_registral_m2"].mask(registral_invalida)

# Impuesto facturado nulo
registrar("Impuesto facturado nulo", (predios["ipu_facturado_2026"].isna() & en_unidad).sum(), "Faltante")

with pd.option_context("display.max_colwidth", 90):
    display(pd.DataFrame(limpieza))

,problema,predios de la unidad,tratamiento
0,Número de pisos igual a 99,1,Faltante
1,Fecha de visita de relleno,8281,Faltante; la visita cuenta como exitosa solo con fecha válida
2,Año de construcción 2000 de relleno,3299,"Faltante, con el indicador anio_sospechoso"
3,Avalúo catastral igual a 0,299,"Faltante, con el indicador avaluo_cero; tarifa efectiva faltante"
4,Área registral igual a 0 o más de 20 veces menor que la construida,985,Faltante
5,Impuesto facturado nulo,32,Faltante


**Interpretación.** El problema más frecuente en la unidad de análisis es la fecha de visita de relleno (8 281 predios), seguido del año de construcción 2000 (3 299) y del área registral inválida (985). Los demás afectan a pocos predios: 299 con avalúo 0, 32 sin impuesto facturado y 1 con 99 pisos.

Todos pasan a faltante. El año 2000 y el avalúo 0 conservan además un indicador, porque el valor de relleno puede decir algo del predio: un levantamiento incompleto en el primer caso y una liquidación con otro avalúo en el segundo.

<a id="caracteristicas"></a>

## 5. Nuevas características

Todas se calculan fila por fila, con información conocida antes del pago de la vigencia 2026, así que se pueden crear antes de partir los datos ([características derivadas](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/caracteristicas-derivadas/)). Ninguna usa la variable objetivo. El historial usa el pago de 2025; el impuesto y la tarifa efectiva usan la facturación de 2026, que es anterior al pago.

| Característica | Cómo se calcula | Por qué |
| --- | --- | --- |
| `historial_2025` | Pagó, no pagó o sin registro de 2025 | La señal más fuerte del pago (V = 0,57); "sin registro" es una categoría propia |
| `novedad_numero_predial` | Novedad frente al catastro anterior; "Sin_Novedad" si no tiene | Distingue predios nuevos y desenglobes entre los que no tienen registro |
| `log_avaluo`, `avaluo_cero` | log10 del avalúo positivo; indicador de avalúo 0 | El avalúo es muy asimétrico y se analiza en escala logarítmica |
| `log_ipu`, `tarifa_efectiva` | log10 del impuesto facturado; impuesto sobre avalúo, en por mil | El monto a pagar y su peso relativo |
| `log_area_construida` | log10(1 + área construida) | Tamaño; vale 0 en los lotes |
| `log_area_registral` | log10 del área registral válida | Tamaño según el título |
| `log_area_terreno` | log10(1 + área de terreno atribuible) | Terreno propio; en la PH, coeficiente de copropiedad por el terreno de la matriz |
| `pisos_edificio` | Pisos de la construcción propia; en la PH, los de la matriz; 0 en los lotes | Altura del edificio |
| `n_unidades`, `n_construcciones`, `anio_construccion` | Conteos y mediana del año de las unidades | Tamaño y antigüedad |
| `visita_exitosa_efectiva` | Visita exitosa con fecha válida, propia o de la matriz | Información de campo confiable |
| `n_titulares`, `log_predios_mismo_titular`, `titular_juridico` | Número de titulares; log10 del mayor número de predios que tiene uno de ellos; si alguno es persona jurídica (sin datos personales) | Quién es el dueño: varios dueños, dueños de muchos predios, empresas |
| `tiene_matricula` | Matrícula inmobiliaria registrada | Formalidad del predio |

Las categóricas restantes (`condicion_predio`, `destinacion_economica`, `zona`, `resultado_visita`, `tipo_derecho`, `tipo_predio`, `autoriza_notificaciones`) pasan como texto y se codifican en el pipeline. El decil de avalúo del notebook 06 no se usa: sus cortes salen de todo el conjunto, y el modelo encuentra los suyos con `log_avaluo`.

In [6]:
por_predio = predios.set_index("id_predio")

# Conteos de unidades y construcciones, y pisos de la construcción más alta
predios["n_unidades"] = predios["id_predio"].map(unidades.groupby("id_predio").size()).fillna(0).astype(int)
predios["n_construcciones"] = predios["id_predio"].map(construcciones.groupby("id_predio").size()).fillna(0).astype(int)
pisos_propios = predios["id_predio"].map(construcciones.groupby("id_predio")["total_pisos"].max())

# Unidades de PH: el terreno y las construcciones están registrados en su matriz
es_unidad_ph = predios["condicion_predio"].eq("PH.Unidad_Predial")
pisos_matriz = predios["id_predio_matriz"].map(pd.Series(pisos_propios.values, index=predios["id_predio"]))
predios["pisos_edificio"] = pisos_propios.where(~es_unidad_ph, pisos_matriz)
predios.loc[~es_unidad_ph & predios["n_construcciones"].eq(0), "pisos_edificio"] = 0
terreno_matriz = predios["id_predio_matriz"].map(por_predio["terreno.area_terreno"])
area_terreno = predios["terreno.area_terreno"].where(~es_unidad_ph, predios["coeficiente_copropiedad"] * terreno_matriz)

# Visita exitosa con fecha válida, propia o de la matriz (etapa 3.3 del notebook 06)
visita_propia = predios["resultado_visita"].eq("Exitoso") & predios["fecha_visita_valida"].notna()
visita_por_id = pd.Series(visita_propia.values, index=predios["id_predio"])
predios["visita_exitosa_efectiva"] = (visita_propia | predios["id_predio_matriz"].map(visita_por_id).eq(True)).astype(int)

# Escalas logarítmicas y razones
avaluo = predios["avaluo_catastral"].where(predios["avaluo_catastral"] > 0)
ipu = predios["ipu_facturado_2026"].where(predios["ipu_facturado_2026"] > 0)
predios["log_avaluo"] = np.log10(avaluo)
predios["log_ipu"] = np.log10(ipu)
predios["tarifa_efectiva"] = ipu / avaluo * 1000
predios["log_area_construida"] = np.log10(1 + predios["area_construida_total"])
predios["log_area_registral"] = np.log10(predios["area_registral_valida"])
predios["log_area_terreno"] = np.log10(1 + area_terreno)

# Categóricas en texto
predios["novedad_numero_predial"] = predios["tipo_novedad_numero_predial"].fillna("Sin_Novedad")
predios["zona"] = predios["zona"].map({"00": "Rural", "01": "Urbano"})
predios["tipo_predio"] = predios["tipo"]
predios["autoriza_notificaciones"] = predios["autoriza_notificaciones"].map({True: "Sí", False: "No"}).fillna("Sin dato")

# Titulares: solo conteos, sin datos personales
predios["n_titulares"] = predios["titulares.n_titulares"]
predios["log_predios_mismo_titular"] = np.log10(predios["titulares.max_predios_mismo_titular"])
predios["titular_juridico"] = predios["titulares.n_persona_juridica"].fillna(0).gt(0).astype(int)
predios["tiene_matricula"] = predios["matricula_inmobiliaria"].notna().astype(int)

# Resumen de las numéricas en la unidad de análisis (sin el impuesto ni la tarifa, que salen de la facturación)
predios.loc[en_unidad, [c for c in um.NUMERICAS if c not in ("log_ipu", "tarifa_efectiva")]].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
log_avaluo,18850.00,8.28,0.70,5.58,7.81,8.36,8.75,11.55
log_area_construida,19149.00,1.79,0.89,0.00,1.15,2.05,2.37,5.27
log_area_registral,18164.00,2.31,0.85,-0.96,1.76,2.27,2.90,6.57
log_area_terreno,18879.00,2.32,0.82,0.10,1.79,2.29,2.87,6.57
n_unidades,19149.00,2.16,2.46,0.00,1.00,2.00,3.00,80.00
n_construcciones,19149.00,0.56,1.60,0.00,0.00,0.00,1.00,62.00
pisos_edificio,19149.00,2.16,1.16,0.00,1.00,2.00,3.00,5.00
anio_construccion,13405.00,2014.60,10.54,1944.00,2010.00,2018.00,2024.00,2025.00
n_titulares,19149.00,1.57,1.85,1.00,1.00,1.00,2.00,88.00
log_predios_mismo_titular,19149.00,0.82,0.91,0.00,0.00,0.48,1.38,2.89


**Interpretación.** Las escalas logarítmicas dejan el avalúo, el impuesto y las áreas en rangos comparables: el log10 del avalúo va de 5,6 a 11,6, con la mitad central entre 7,8 y 8,8. Con la información de la matriz, el área de terreno queda con 1,4 % de faltantes y los pisos del edificio sin faltantes. El número de predios del mismo titular tiene una cola muy larga (mediana de 3 y máximo de 770), por eso entra en log10.

El año de construcción es la única variable con muchos faltantes (30 %): son los lotes sin construir y el año 2000 de relleno.

<a id="transformaciones"></a>

## 6. Codificación, escalado e imputación

Estas transformaciones aprenden de los datos (medianas, medias y desviaciones, categorías), así que se ajustan solo con el conjunto de entrenamiento, dentro del pipeline del notebook 10 ([pipeline](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/pipeline/)). Aquí se definen: las listas de columnas y la función `crear_preprocesador` están en [utilidades_modelado.py](utilidades_modelado.py), para que el notebook 10 y el producto usen exactamente las mismas.

| Tipo | Columnas | Tratamiento |
| --- | --- | --- |
| Numéricas | `um.NUMERICAS` | Imputación con la mediana y una columna indicadora por cada variable con faltantes ([valores nulos](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/nulos/)); estandarización para la regresión logística ([estandarizar](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/estandarizar/)) |
| Categóricas | `um.CATEGORICAS` | One-hot ([one-hot](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/one-hot/)); las categorías con menos de 30 predios se agrupan en una sola, y las que no aparecen en entrenamiento quedan en ceros |
| Binarias | `um.BINARIAS` | Sin cambios (ya son 0 y 1) |

Dos diferencias con las guías, a propósito:

- **No se elimina la primera categoría del one-hot** (`drop_first`). La regresión logística del notebook 10 va regularizada, así que las columnas redundantes no la vuelven inestable, y conservarlas hace más fácil leer los coeficientes por categoría.
- **La imputación agrega un indicador de faltante.** En estos datos el faltante tiene significado (un lote no tiene año de construcción; un predio sin impuesto no se liquidó), y el indicador le permite al modelo usarlo.

In [7]:
tipos = pd.DataFrame(
    [(c, "numérica") for c in um.NUMERICAS] + [(c, "categórica") for c in um.CATEGORICAS] +
    [(c, "binaria") for c in um.BINARIAS], columns=["columna", "tipo"]).set_index("columna")
tipos["% faltante"] = predios.loc[en_unidad, tipos.index].isna().mean() * 100
tipos["categorías"] = [predios.loc[en_unidad, c].nunique() if t != "numérica" else None for c, t in tipos["tipo"].items()]
tipos.round(1)

,tipo,% faltante,categorías
columna,,,
log_avaluo,numérica,1.60,NaN
log_ipu,numérica,0.20,NaN
tarifa_efectiva,numérica,1.60,NaN
log_area_construida,numérica,0.00,NaN
log_area_registral,numérica,5.10,NaN
log_area_terreno,numérica,1.40,NaN
n_unidades,numérica,0.00,NaN
n_construcciones,numérica,0.00,NaN
pisos_edificio,numérica,0.00,NaN


**Interpretación.** Todas las categóricas están completas. La mayoría tiene pocas categorías, salvo la destinación económica (24), con 15 categorías de menos de 30 predios que el one-hot agrupa en una sola. En `autoriza_notificaciones`, "Sin dato" es la categoría más frecuente: corresponde a los predios sin nadie que atendiera la visita.

Solo el año de construcción pasa del 5 % de faltantes. Se imputa con la mediana, y su indicador de faltante conserva la diferencia entre los lotes y los predios construidos.

<a id="evidencia"></a>

## 7. Evidencia de los datos preparados

Se aplica la unidad de análisis de la etapa 3 y se guardan el identificador, la variable objetivo, el impuesto facturado (para el recaudo esperado del notebook 10) y las características. Después: dimensiones, tipos, faltantes, la variable objetivo por grupo y una muestra sin columnas reservadas ni variables que salgan de ellas.

In [8]:
columnas = ["id_predio", um.OBJETIVO, "ipu_facturado_2026"] + um.CARACTERISTICAS
preparados = predios.loc[en_unidad, columnas].reset_index(drop=True)
preparados[um.OBJETIVO] = preparados[um.OBJETIVO].astype(int)

# Comprobaciones: un predio por fila, sin faltantes en el objetivo ni en las categóricas
assert preparados["id_predio"].is_unique
assert preparados[um.OBJETIVO].notna().all() and preparados[um.CATEGORICAS].notna().all().all()

print(f"Datos preparados: {preparados.shape[0]:,} predios x {preparados.shape[1]} columnas "
      f"({len(um.NUMERICAS)} numéricas, {len(um.CATEGORICAS)} categóricas, {len(um.BINARIAS)} binarias)")
print(f"Pagó 2026: {preparados[um.OBJETIVO].mean():.1%}")
display(preparados.dtypes.value_counts().rename("columnas").to_frame())

faltantes = preparados[um.CARACTERISTICAS].isna().mean().mul(100)
display(faltantes[faltantes > 0].sort_values(ascending=False).round(1).rename("% faltante").to_frame())

# Muestra: sin el identificador, sin columnas reservadas y sin las que salen de la facturación
derivadas_reservadas = ["log_ipu", "tarifa_efectiva", um.GRUPO, um.OBJETIVO, "ipu_facturado_2026", "id_predio"]
# Tampoco los conteos de titulares, y los logaritmos con un decimal, para que ninguna fila ubique un predio
titulares = ["n_titulares", "log_predios_mismo_titular", "titular_juridico"]
muestra = preparados.drop(columns=derivadas_reservadas + titulares).sample(5, random_state=um.SEMILLA)
muestra.reset_index(drop=True).round({c: 1 for c in muestra.columns if c.startswith("log_")})

Datos preparados: 19,149 predios x 29 columnas (12 numéricas, 9 categóricas, 5 binarias)
Pagó 2026: 64.0%


,columnas
float64,10
int64,9
object,9
int32,1


,% faltante
anio_construccion,30.00
log_area_registral,5.10
tarifa_efectiva,1.60
log_avaluo,1.60
log_area_terreno,1.40
log_ipu,0.20


,log_avaluo,log_area_construida,log_area_registral,log_area_terreno,n_unidades,n_construcciones,pisos_edificio,anio_construccion,novedad_numero_predial,condicion_predio,destinacion_economica,zona,resultado_visita,tipo_derecho,tipo_predio,autoriza_notificaciones,visita_exitosa_efectiva,avaluo_cero,anio_sospechoso,tiene_matricula
0,9.20,2.70,3.30,3.20,2,0,2.00,2024.00,Cambio_Numero_Predial,PH.Unidad_Predial,Habitacional,Rural,Incompleto,Dominio,Predio.Privado,Sin dato,1,0,0,1
1,8.50,2.20,2.00,2.00,3,0,3.00,NaN,Cambio_Numero_Predial,PH.Unidad_Predial,Habitacional,Urbano,Incompleto,Dominio,Predio.Privado,Sin dato,0,0,1,1
2,8.60,2.50,2.70,2.70,4,3,2.00,2010.00,Sin_Novedad,NPH,Habitacional,Rural,No_Permitieron_Acceso,Dominio,Predio.Privado,No,0,0,0,1
3,8.00,0.00,2.30,2.30,0,0,0.00,NaN,Predio_Nuevo,NPH,Lote_Urbanizado_No_Construido,Urbano,No_Hay_Nadie,Dominio,Predio.Privado,No,0,0,0,1
4,6.20,0.50,0.10,0.40,1,0,3.00,2025.00,Predio_Nuevo,PH.Unidad_Predial,Habitacional,Urbano,Incompleto,Dominio,Predio.Privado,Sin dato,1,0,0,1


In [9]:
# Forma de la matriz que entra al modelo: el preprocesador se ajusta aquí solo para mostrar
# sus columnas de salida; el notebook 10 lo ajusta únicamente con entrenamiento
forma = um.crear_preprocesador().fit(preparados[um.CARACTERISTICAS])
nombres = pd.Series(forma.get_feature_names_out())
print(f"Columnas que recibe el modelo: {len(nombres)}")
nombres.str.split("__").str[0].value_counts().rename("columnas").to_frame()

Columnas que recibe el modelo: 63


,columnas
categoricas,40
numericas,18
binarias,5


In [10]:
um.CARPETA_MODELADO.mkdir(parents=True, exist_ok=True)
preparados.to_parquet(um.ARCHIVO_PREPARADO, index=False)
print(f"Guardado en {um.ARCHIVO_PREPARADO.relative_to(um.RAIZ)}")

Guardado en datos\modelado\predios_preparados.parquet


**Interpretación.** El archivo preparado tiene 19 149 predios y 29 columnas: el identificador, la variable objetivo, el impuesto facturado (solo para el recaudo esperado del notebook 10) y 26 características. Seis numéricas tienen faltantes, todas por debajo del 5,1 % salvo el año de construcción. Con el one-hot y los indicadores de faltante, el modelo recibe 63 columnas: 40 de las categóricas, 18 de las numéricas (12 variables y 6 indicadores) y las 5 binarias.

<a id="diagrama"></a>

## 8. Diagrama de bloques

El diagrama, en Mermaid, está en [../diagramas/preparacion.mmd](../diagramas/preparacion.mmd). Resume el flujo:

1. **Carga:** el archivo de predios, con las construcciones y las unidades desanidadas.
2. **Unidad de análisis:** los 19 149 predios facturados en 2026, sin vías, bienes de uso público ni parque cementerio; objetivo `pago_2026`.
3. **Limpieza:** los valores de relleno y las incoherencias pasan a faltantes, con indicadores donde el faltante tiene significado.
4. **Nuevas características:** historial de 2025, novedad del número predial, escalas logarítmicas, área de terreno y pisos tomados de la matriz en la PH, visita efectiva y conteos de titulares.
5. **Archivo preparado:** `datos/modelado/predios_preparados.parquet`.
6. **Partición** estratificada por pago e historial (notebook 09).
7. **Pipeline** ajustado con entrenamiento: imputación, one-hot y estandarización, y el modelo (notebook 10).